# KalKalori — Simulation Smoke Test

Notebook version of `core/tests/simulation_smoke.py`. Exercises
`BareTubeHeatExchanger.simulate(...)` (v0.5.x mean-property simulation)
WITHOUT CoolProp / IAPWS / PsychroLib: it uses a tiny linear
temperature-dependent provider plus `ConstantPropertyProvider` so the outer
iteration loop, the forced-averaged single-pass short-circuit, convergence
diagnostics, non-convergence handling, and `surface_margin` derating can all
be exercised without external property backends.

For Rating (closing a known heat balance to get overdesign/margin), see
`core/tests/heat_balance_rating_smoke.py`.

In [1]:
from pathlib import Path
import sys
from dataclasses import dataclass

import pandas as pd

# Make the notebook usable both from repository root and from core/tests.
cwd = Path.cwd().resolve()
candidate_roots = [cwd, *cwd.parents]

for candidate in candidate_roots:
    if (candidate / "core").is_dir():
        workspace_root = candidate
        break
else:
    raise RuntimeError("Could not find repository root containing the 'core' directory.")

if str(workspace_root) not in sys.path:
    sys.path.insert(0, str(workspace_root))

print("Workspace root:", workspace_root)

Workspace root: C:\Users\pawel\GitHub\kalkalori


In [2]:
from core.geometry.tube import BareTube
from core.geometry.bundle import TubeBundle
from core.properties.common import FluidTransportProperties
from core.properties.fluids import ConstantPropertyProvider
from core.models.bare_tube import BareTubeHeatExchanger
from core.models.simulation import HXSideInput

print("Imports completed.")

Imports completed.


## Setup: Toy Provider, Geometry, Helpers

`LinearGasProvider` is not physical: it only needs to make properties move
with temperature so the mean-property iteration produces a visibly different
result from an inlet-only evaluation. Ignores pressure.

In [3]:
@dataclass(frozen=True)
class LinearGasProvider:
    rho0: float
    mu0: float
    k0: float
    cp0: float
    T_ref: float = 300.0
    rho_slope: float = -1.0 / 300.0   # rho falls with T (ideal-gas-like)
    mu_slope: float = 1.5e-3
    k_slope: float = 1.5e-3
    cp_slope: float = 2.0e-4

    def at(self, T: float, p: float) -> FluidTransportProperties:
        dT = T - self.T_ref
        rho = max(self.rho0 * (1.0 + self.rho_slope * dT), 1e-3)
        mu = max(self.mu0 * (1.0 + self.mu_slope * dT), 1e-7)
        k = max(self.k0 * (1.0 + self.k_slope * dT), 1e-4)
        cp = max(self.cp0 * (1.0 + self.cp_slope * dT), 1.0)
        return FluidTransportProperties(rho=rho, mu=mu, k=k, cp=cp)


def build_bundle() -> TubeBundle:
    """Geometry from the v0.4.5 gas-gas test case."""
    tube = BareTube(
        D_i=25e-3 - 2 * 1.5e-3,
        D_o=25e-3,
        length_total=2.8,
        length_effective=2.8,
        wall_k=50.0,
    )
    return TubeBundle(
        tube=tube,
        n_rows=36,
        n_tubes_per_row=56,
        pitch_transverse=35e-3,
        pitch_longitudinal=35e-3,
        layout="staggered",
        n_passes_tube=2,
        flow_arrangement="counterflow",
    )


def c_to_k(t_c: float) -> float:
    return t_c + 273.15


def kgh_to_kgs(m: float) -> float:
    return m / 3600.0


hx = BareTubeHeatExchanger(build_bundle())

inside_var = HXSideInput(
    provider=LinearGasProvider(rho0=1.13, mu0=1.9e-5, k0=0.027, cp0=1007.0),
    m_dot=kgh_to_kgs(18_220.0), T_in=c_to_k(30.0), p=101_325.0,
)
outside_var = HXSideInput(
    provider=LinearGasProvider(rho0=0.50, mu0=3.1e-5, k0=0.052, cp0=1180.0),
    m_dot=kgh_to_kgs(28_380.0), T_in=c_to_k(400.0), p=101_325.0,
)

print("Geometry and variable-property side inputs ready.")

Geometry and variable-property side inputs ready.


## Case 1 — Default `simulate()`: Mean-Property Iteration (Variable Properties)

In [4]:
res = hx.simulate(inside_var, outside_var)

assert res.converged and res.iterations > 1, "case 1 should iterate and converge"

pd.Series({
    "converged": res.converged,
    "iterations": res.iterations,
    "residual_q_rel": res.residual_q_rel,
    "q_kW": res.q / 1e3,
    "UA_W_K": res.UA,
    "U_mean_W_m2K": res.U_mean,
    "T_mean_inside_C": res.T_mean_inside - 273.15,
    "T_mean_outside_C": res.T_mean_outside - 273.15,
    "T_out_inside_C": res.T_out_inside - 273.15,
    "T_out_outside_C": res.T_out_outside - 273.15,
}, name="value").to_frame()

,value
converged,True
iterations,14
residual_q_rel,0.0
q_kW,1646.846627
UA_W_K,14354.757872
U_mean_W_m2K,32.378553
T_mean_inside_C,186.564585
T_mean_outside_C,316.326582
T_out_inside_C,343.12917
T_out_outside_C,232.653163


## Case 2 — Forced Averaged Properties (`ConstantPropertyProvider`) → 1 Pass

In [5]:
inside_c = HXSideInput(
    provider=ConstantPropertyProvider(
        FluidTransportProperties(rho=1.13, mu=1.9e-5, k=0.027, cp=1007.0)
    ),
    m_dot=kgh_to_kgs(18_220.0), T_in=c_to_k(30.0), p=101_325.0,
)
outside_c = HXSideInput(
    provider=ConstantPropertyProvider(
        FluidTransportProperties(rho=0.50, mu=3.1e-5, k=0.052, cp=1180.0)
    ),
    m_dot=kgh_to_kgs(28_380.0), T_in=c_to_k(400.0), p=101_325.0,
)

res2 = hx.simulate(inside_c, outside_c)

assert res2.converged and res2.iterations > 1, "case 2 must iterate for wall correction"

pd.Series({
    "converged": res2.converged,
    "iterations": res2.iterations,
    "q_kW": res2.q / 1e3,
    "T_out_inside_C": res2.T_out_inside - 273.15,
    "T_out_outside_C": res2.T_out_outside - 273.15,
}, name="value").to_frame()

,value
converged,True
iterations,12
q_kW,1564.116836
T_out_inside_C,336.897852
T_out_outside_C,231.85758


## Case 3 — `iterate=False` Escape Hatch (Variable Properties → Inlet-Only, 1 Pass)

In [6]:
res_inlet = hx.simulate(inside_var, outside_var, iterate=False)

assert res_inlet.converged and res_inlet.iterations == 1, "iterate=False -> single pass"

dq = (res.q - res_inlet.q) / res_inlet.q * 100.0
print(f"mean-property vs inlet-only duty shift: {dq:+.2f}%")

pd.Series({
    "converged": res_inlet.converged,
    "iterations": res_inlet.iterations,
    "q_kW": res_inlet.q / 1e3,
}, name="value").to_frame()

mean-property vs inlet-only duty shift: +1.74%


,value
converged,True
iterations,1
q_kW,1618.621718


## Case 4 — Force Non-Convergence (`max_iter=3`) → Warning, No Hang

In [7]:
res4 = hx.simulate(inside_var, outside_var, max_iter=3, relaxation_factor=0.2)

assert not res4.converged and res4.iterations == 3, "case 4 must not converge"

print(f"converged / iterations : {res4.converged} / {res4.iterations}")
for w in (res4.warnings or []):
    if w.source == "simulation":
        print(f"warning[{w.severity}] {w.code}")

converged / iterations : False / 3
warning[warning] simulation_not_converged


## Case 5 — `surface_margin=0.0` Must Reproduce Case 2 Bit-for-Bit

In [8]:
res_margin0 = hx.simulate(inside_c, outside_c, surface_margin=0.0)

assert res_margin0.q == res2.q, "surface_margin=0.0 must reproduce q bit-for-bit"
assert res_margin0.T_out_inside == res2.T_out_inside
assert res_margin0.T_out_outside == res2.T_out_outside
assert res_margin0.Q_full == res_margin0.Q_derated == res_margin0.q

pd.Series({
    "q_kW": res_margin0.q / 1e3,
    "Q_full_kW": res_margin0.Q_full / 1e3,
    "Q_derated_kW": res_margin0.Q_derated / 1e3,
    "Q_full == Q_derated": res_margin0.Q_full == res_margin0.Q_derated,
}, name="value").to_frame()

,value
q_kW,1564.116836
Q_full_kW,1564.116836
Q_derated_kW,1564.116836
Q_full == Q_derated,True


## Case 6 — `surface_margin` > 0 Derates Duty Monotonically

In [9]:
res_margin_low = hx.simulate(inside_c, outside_c, surface_margin=0.2)
res_margin_high = hx.simulate(inside_c, outside_c, surface_margin=0.5)

assert res_margin_low.q < res_margin0.q, "margin=0.2 must derate duty below margin=0"
assert res_margin_high.q < res_margin_low.q, "margin=0.5 must derate duty further than 0.2"
assert res_margin_high.Q_full > res_margin_high.Q_derated, "Q_full must exceed Q_derated when margin > 0"
assert res_margin_low.Q_full > res_margin_low.Q_derated

pd.DataFrame([
    {"surface_margin": 0.0, "q_kW": res_margin0.q / 1e3},
    {"surface_margin": 0.2, "q_kW": res_margin_low.q / 1e3},
    {"surface_margin": 0.5, "q_kW": res_margin_high.q / 1e3},
])

,surface_margin,q_kW
0,0.0,1564.116836
1,0.2,1477.289280
2,0.5,1360.462285


## Energy-Balance Sanity Check (Case 1)

In [10]:
q_in = inside_var.m_dot * res.inside_props_mean.cp * (res.T_out_inside - inside_var.T_in)
q_out = outside_var.m_dot * res.outside_props_mean.cp * (outside_var.T_in - res.T_out_outside)

print(f"energy-balance: q_in={q_in/1e3:.2f} kW  q_out={q_out/1e3:.2f} kW  q={res.q/1e3:.2f} kW")

assert abs(q_in - res.q) / res.q < 0.02
assert abs(q_out - res.q) / res.q < 0.02

print("\nALL SMOKE CHECKS PASSED")

energy-balance: q_in=1646.85 kW  q_out=1646.85 kW  q=1646.85 kW

ALL SMOKE CHECKS PASSED


## Inlet, midpoint, and outlet fluid properties

Point states below come directly from the solver's existing hydraulic results. The wet midpoint uses arithmetic mean temperature and water ratio; no transport-property provider is called for presentation.

In [11]:
import math
import pandas as pd


def endpoint_property_table(solver_result, side):
    """Read solver-owned hydraulic point states without provider calls."""
    states = (
        ("inlet", getattr(solver_result, f"{side}_properties_inlet")),
        ("midpoint", getattr(solver_result, f"{side}_properties_midpoint")),
        ("outlet", getattr(solver_result, f"{side}_properties_outlet")),
    )
    return pd.DataFrame(
        [
            {
                "state": name,
                "T [°C]": state.T - 273.15,
                "p [Pa]": state.p,
                "rho [kg/m³]": state.rho,
                "cp [J/(kg·K)]": state.cp,
                "mu [Pa·s]": state.mu,
                "k [W/(m·K)]": state.k,
                "Pr [-]": state.Pr,
            }
            for name, state in states
            if state is not None
        ]
    ).set_index("state")


def representative_0d_property_table(solver_result):
    """Keep representative thermal properties separate from point states."""
    if hasattr(solver_result, "inside_props_mean"):
        pairs = (
            ("inside", solver_result.T_mean_inside, solver_result.inside_props_mean),
            ("outside", solver_result.T_mean_outside, solver_result.outside_props_mean),
        )
    elif getattr(solver_result, "thermal_state", None) is not None:
        thermal = solver_result.thermal_state
        pairs = (
            ("inside", thermal.inside_bulk_temperature, thermal.inside_bulk_props),
            ("outside", thermal.outside_bulk_temperature, thermal.outside_bulk_props),
        )
    else:
        return pd.DataFrame()
    return pd.DataFrame(
        [
            {
                "side": side,
                "T representative [°C]": temperature - 273.15,
                "rho [kg/m³]": props.rho,
                "cp [J/(kg·K)]": props.cp,
                "mu [Pa·s]": props.mu,
                "k [W/(m·K)]": props.k,
                "Pr [-]": props.mu * props.cp / props.k,
            }
            for side, temperature, props in pairs
        ]
    ).set_index("side")


def wet_gas_state_table(solver_result, outside_provider_for_result=None):
    """Combine hydraulic states with wet diagnostics already returned by the solver."""
    pc = getattr(solver_result, "outside_phase_change", None)
    if pc is None or not pc.capable or pc.W_in is None:
        return pd.DataFrame()

    W_mid = 0.5 * (pc.W_in + pc.W_out)
    dew_mid = math.nan
    if outside_provider_for_result is not None:
        from core.phase_change.capability import detect_phase_change_capability
        from core.phase_change.integration import _dew_point_at_ratio

        capability = detect_phase_change_capability(outside_provider_for_result)
        midpoint_state = solver_result.outside_properties_midpoint
        dew_mid_value = _dew_point_at_ratio(
            capability, W_mid, p=midpoint_state.p
        )
        dew_mid = math.nan if dew_mid_value is None else dew_mid_value

    dry_flow = pc.m_dot_dry_carrier
    vapor_mid = (
        math.nan
        if dry_flow is None
        else dry_flow * W_mid
    )
    gas_mid = (
        math.nan
        if dry_flow is None
        else dry_flow + vapor_mid
    )
    values = (
        ("inlet", pc.W_in, pc.dew_point_in, pc.m_dot_gas_in, pc.m_dot_water_vapor_in),
        ("midpoint", W_mid, dew_mid, gas_mid, vapor_mid),
        ("outlet", pc.W_out, pc.dew_point_out, pc.m_dot_gas_out, pc.m_dot_water_vapor_out),
    )
    return pd.DataFrame(
        [
            {
                "state": name,
                "W [kg/kg dry]": W,
                "dew point [°C]": (
                    math.nan if dew_point is None else dew_point - 273.15
                ),
                "m_dot gas [kg/s]": gas_flow,
                "m_dot water vapor [kg/s]": vapor_flow,
            }
            for name, W, dew_point, gas_flow, vapor_flow in values
        ]
    ).set_index("state")


def condensation_summary_table(solver_result):
    pc = getattr(solver_result, "outside_phase_change", None)
    if pc is None or not pc.capable:
        return pd.DataFrame()
    return pd.DataFrame(
        [
            {
                "m_dot condensate [kg/s]": pc.m_dot_condensate,
                "Q_sensible [W]": pc.Q_sensible,
                "Q_latent [W]": pc.Q_latent,
                "wet_surface_fraction [-]": pc.wet_surface_fraction,
                "wall Tmin [°C]": (
                    math.nan if pc.wall_temperature_min is None
                    else pc.wall_temperature_min - 273.15
                ),
                "wall Tmean [°C]": (
                    math.nan if pc.wall_temperature_mean is None
                    else pc.wall_temperature_mean - 273.15
                ),
                "wall Tmax [°C]": (
                    math.nan if pc.wall_temperature_max is None
                    else pc.wall_temperature_max - 273.15
                ),
            }
        ],
        index=["outside"],
    )

In [12]:
endpoint_results = [('Variable-property Simulation', res)]
outside_provider_for_endpoint_table = None

for result_label, endpoint_result in endpoint_results:
    print(result_label)
    print("Inside")
    display(endpoint_property_table(endpoint_result, "inside"))
    print("Outside")
    display(endpoint_property_table(endpoint_result, "outside"))

    wet_table = wet_gas_state_table(
        endpoint_result, outside_provider_for_endpoint_table
    )
    if not wet_table.empty:
        print("Outside wet-gas mass and dew-point diagnostics")
        display(wet_table)
        display(condensation_summary_table(endpoint_result))

Variable-property Simulation
Inside


,T [°C],p [Pa],rho [kg/m³],cp [J/(kg·K)],mu [Pa·s],k [W/(m·K)],Pr [-]
state,,,,,,,
inlet,30.000000,101325.0,1.118135,1007.634410,0.000019,0.027128,0.709076
midpoint,186.564585,101325.0,0.528408,1039.166517,0.000024,0.033468,0.731265
outlet,343.129170,101325.0,0.001000,1070.698625,0.000028,0.039809,0.753455


Outside


,T [°C],p [Pa],rho [kg/m³],cp [J/(kg·K)],mu [Pa·s],k [W/(m·K)],Pr [-]
state,,,,,,,
inlet,400.000000,101325.0,0.001000,1268.063400,0.000048,0.081106,0.755961
midpoint,316.326582,101325.0,0.017539,1248.316473,0.000044,0.074579,0.744189
outlet,232.653163,101325.0,0.156995,1228.569546,0.000041,0.068053,0.732416


## Representative 0D properties used by the solver

These lumped thermal-model properties are retained separately; they are not substitutes for inlet or outlet states.

In [13]:
for result_label, endpoint_result in endpoint_results:
    representative_table = representative_0d_property_table(endpoint_result)
    if not representative_table.empty:
        print(result_label)
        display(representative_table)

Variable-property Simulation


,T representative [°C],rho [kg/m³],cp [J/(kg·K)],mu [Pa·s],k [W/(m·K)],Pr [-]
side,,,,,,
inside,186.564585,0.528471,1039.163192,0.000024,0.033468,0.731263
outside,316.326582,0.017518,1248.319504,0.000044,0.074580,0.744190
